&nbsp;

&nbsp;

# Enhancer-derived lncRNA (down5p)

- enhancer の近傍にある lncRNA を調査する
- [ENCODE-rE2G](https://github.com/EngreitzLab/ENCODE_rE2G) の予測結果に該当する遺伝子を調査する
- これまでの定義では、紐づく fanta.bio の CRE が `ELA` であることを条件としていたが、enhancer から転写されるとされる `ENEMAL` の近傍にある CRE が `PLA` であり、見逃していることに気づいた
- このような背景から、[ENCODE-rE2G](https://github.com/EngreitzLab/ENCODE_rE2G) の予測結果とオーバーラップするデータを、enhancer-derived lncRNA として調査する
- [../05_ENCODE_rE2G/04_encode_re2g_parse_ctrl.ipynb](../05_ENCODE_rE2G/04_encode_re2g_parse_ctrl.ipynb)で処理した結果と、down5p transcript list を `join`する

In [1]:
import datetime
import os
import pathlib

import polars as pl
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

&nbsp;

## 1. e2g dataset (Non stress)

- `TargetGene` の近傍ではない enhancer を標的とする
- `isSelfPromoter` == False でフィルタリングする

In [2]:
e2g_join_filter = pl.scan_parquet(
    "../../out/ENCODE_rE2G/parquet_file/encode_re2g_ctrl_encode_predictions_v261006.parquet"
).filter(
    pl.col("isSelfPromoter") == False
)

display(e2g_join_filter.head(5).collect())

ENCODE-rE2G.Score,re2g_dataset,Model,sample_info,cre_id,cre_name,cre_chr,fantabio_cre_class,enhancer_cre_overlap_bp,enhancer_cre_overlap_frac_of_fantabio_cre,TargetGene,target_cre_id,target_cre_name,target_fantabio_cre_class,class,isSelfPromoter,distanceToTSS,re2g_enhancer_locus,fantabio_enhancer_cre,target_tss_info,target_cre_info,target_cre_overlap_info
f64,cat,cat,struct[4],str,str,cat,str,i64,f64,str,str,str,str,cat,bool,f64,struct[3],struct[3],struct[5],struct[10],struct[2]
0.824621,"""ENCSR045WCT_ENCFF292SIC""","""DNase-only""","{""GM18507_ENCSR000EJI"",""GM18507"",""cell line"",""EFO:0002787""}","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",329,0.691176,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""genic""",false,760.0,"{""chr1"",961096,961596}","{960949,961425,-0.483871}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.813417,"""ENCSR157DJQ_ENCFF901GNX""","""DNase-only""","{""iPS-NIHi7_ENCSR000EKK"",""iPS-NIHi7"",""cell line"",""EFO:0007953""}","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",313,0.657563,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""genic""",false,776.0,"{""chr1"",961112,961612}","{960949,961425,-0.483871}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.595841,"""ENCSR167ECG_ENCFF947UAT""","""DNase-only""","{""HAP-1_ENCSR620QNS"",""HAP-1"",""cell line"",""EFO:0007598""}","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",261,0.548319,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""genic""",false,828.0,"{""chr1"",961164,961664}","{960949,961425,-0.483871}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.72318,"""ENCSR213XEF_ENCFF312ZPL""","""DNase-only""","{""common_myeloid_progenitor__CD34-positive_ENCSR122VUW"",""common myeloid progenitor, CD34-positive"",""primary cell"",""CL:0001059""}","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",274,0.57563,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""genic""",false,815.0,"{""chr1"",961151,961651}","{960949,961425,-0.483871}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.558777,"""ENCSR221FYS_ENCFF431CEI""","""DNase-only""","{""L1-S8_ENCSR314EZY"",""L1-S8"",""cell line"",""EFO:0007599""}","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",280,0.588235,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""genic""",false,809.0,"{""chr1"",961145,961645}","{960949,961425,-0.483871}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"


In [3]:
e2g_join_filter_unique_count = e2g_join_filter.group_by(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "target_cre_id",
    "target_cre_name",
    "target_fantabio_cre_class"
).agg(
    pl.col("re2g_dataset").unique().alias("re2g_dataset"),
    pl.col("re2g_dataset").n_unique().alias("re2g_dataset_count"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").alias("enhancer_cre_overlap_frac_of_fantabio_cre"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").min().alias("enhancer_cre_overlap_frac_of_fantabio_cre_min"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").max().alias("enhancer_cre_overlap_frac_of_fantabio_cre_max"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").mean().alias("enhancer_cre_overlap_frac_of_fantabio_cre_mean"), # 該当するデータセットで同定された enhancer と fantabio の CRE の重複率の平均値

    pl.col("ENCODE-rE2G.Score").alias("ENCODE-rE2G.Score"),
    pl.col("ENCODE-rE2G.Score").min().alias("ENCODE-rE2G.Score_min"),
    pl.col("ENCODE-rE2G.Score").max().alias("ENCODE-rE2G.Score_max"),
    pl.col("ENCODE-rE2G.Score").mean().alias("ENCODE-rE2G.Score_mean"), # 該当するデータセットで同定された enhancer と target promoter の相互作用のスコアの平均値

    # pl.col("ENCODE-rE2G.Score").quantile(0.25).alias("ENCODE-rE2G.Score_q1"),
    # pl.col("ENCODE-rE2G.Score").median().alias("ENCODE-rE2G.Score_median"),
    # pl.col("ENCODE-rE2G.Score").quantile(0.75).alias("ENCODE-rE2G.Score_q3"),
    # pl.col("ENCODE-rE2G.Score").std().alias("ENCODE-rE2G.Score_std"),
).sort(
    by=["re2g_dataset_count"],
    descending=True
).collect(
    engine="streaming"
)

print(f"ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level): {e2g_join_filter_unique_count.select(['cre_id', 'target_cre_id']).n_unique()}")
display(e2g_join_filter_unique_count)

ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level): 1527047


cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean
str,str,str,str,str,str,list[cat],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64
"""FCHS_114763""","""FCHS_114763""","""PLA""","""FCHS_114759""","""cp3@MYCBP2""","""PLA""","[""ENCSR000MZA_ENCFF654AMU"", ""ENCSR002YWN_ENCFF435DPV"", … ""ENCSR999YQC_ENCFF625FFH""]",1460,"[1.0, 1.0, … 1.0]",0.824242,1.0,0.998917,"[0.987368, 0.988221, … 0.954474]",0.633907,0.997872,0.983285
"""FCHS_359520""","""cp5@GCLC""","""PLA""","""FCHS_359518""","""cp3@GCLC""","""PLA""","[""ENCSR000MZA_ENCFF654AMU"", ""ENCSR002YWN_ENCFF435DPV"", … ""ENCSR999YQC_ENCFF625FFH""]",1460,"[0.8, 0.818349, … 1.0]",0.721101,1.0,0.8565,"[0.995265, 0.994528, … 0.993794]",0.861669,0.99779,0.99017
"""FCHS_172674""","""FCHS_172674""","""PLA""","""FCHS_172678""","""cp2@NR1D1""","""PLA""","[""ENCSR000MZA_ENCFF654AMU"", ""ENCSR002YWN_ENCFF435DPV"", … ""ENCSR999YQC_ENCFF625FFH""]",1460,"[0.783699, 0.783699, … 0.783699]",0.609718,1.0,0.800712,"[0.987729, 0.970177, … 0.98429]",0.329063,0.99659,0.956294
"""FCHS_26739""","""FCHS_26739""","""PLA""","""FCHS_26765""","""cp1@IQGAP3""","""PLA""","[""ENCSR000MZA_ENCFF654AMU"", ""ENCSR002YWN_ENCFF435DPV"", … ""ENCSR999YQC_ENCFF625FFH""]",1460,"[1.0, 1.0, … 0.880531]",0.767699,1.0,0.980355,"[0.95048, 0.889279, … 0.74397]",0.234617,0.980875,0.867362
"""FCHS_26739""","""FCHS_26739""","""PLA""","""FCHS_26728""","""cp1@MEF2D""","""PLA""","[""ENCSR000MZA_ENCFF654AMU"", ""ENCSR002YWN_ENCFF435DPV"", … ""ENCSR999YQC_ENCFF625FFH""]",1460,"[1.0, 1.0, … 0.880531]",0.767699,1.0,0.980355,"[0.97879, 0.944927, … 0.915287]",0.417009,0.995714,0.958852
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_171445""","""FCHS_171445""","""ELA""","""FCHS_171477""","""cp2@CCL14""","""PLA""","[""ENCSR424MBB_ENCFF469OTZ""]",1,[1.0],1.0,1.0,1.0,[0.234971],0.234971,0.234971,0.234971
"""FCHS_544446""","""FCHS_544446""","""PLA""","""FCHS_295310""","""cp1@ACP3""","""PLA""","[""ENCSR997TVR_ENCFF547ARY""]",1,[1.0],1.0,1.0,1.0,[0.721841],0.721841,0.721841,0.721841
"""FCHS_200653""","""FCHS_200653""","""ELA""","""FCHS_202079""","""cp2@CYP4F3""","""PLA""","[""ENCSR621BIJ_ENCFF647VTW""]",1,[1.0],1.0,1.0,1.0,[0.372632],0.372632,0.372632,0.372632


&nbsp;

&nbsp;

&nbsp;

## 2. down5p transcript list

In [4]:
down5p_fantabio = pl.read_parquet(
    "../../out/fantabio/parquet_file/bottom5p_transcript_fantabio_v261001.parquet"
)

pl.Config.set_tbl_rows(2)
display(down5p_fantabio)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000834029.1""",69,117,220,-48,47,56,148,-9,"""EXO5-DT""","""HGNC:55647""","""lncRNA""","""ENSG00000238287.2""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""-""",4,"""URS0002A1647B""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_11684""","""chr1""",40508315,"""40508650""","""cp5@EXO5""","[""ENST00000925740.1""]","""0""",-63,"[{""EH38E2805480"",""PLS""}]","[{""ZNF687"",3399,[{""SRX10478566"",3399}, {""SRX10478565"",2503}]}, {""TCF7"",2966,[{""SRX10774913"",2966}, {""SRX10774914"",2622}]}, … {""NFATC1"",515,[{""SRX335071"",515}]}]","[{""HS_00004908"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}, {""HS_00004909"",""Foreskin Melanocyte Primary Cells skin01"",""Prepuce"",""Skin"",""SVM""}, … {""HS_00004912"",""Small Intestine"",""Small intestine"",""Intestine"",""SVM""}]","[{""hg19::chr1:40974317..40974346,-;hg_11928.1"",""hg_11928.1""}]",null,0.952663,"""ELA""","""255,202,74""","""yellow"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000549545.1""",11,265,130,-254,8,174,69,-166,"""COQ10A""","""HGNC:26515""","""retained_intron""","""ENSG00000135469.15""",[],"""+""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_96252""","""chr12""",56268140,"""56268446""","""cp6@COQ10A""","[""ENST00000549545.1""]","""-23""",-23,"[{""EH38E4064794"",""pELS""}]",null,"[{""HS_00173371"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_02053667"",""Thyroid Gland"",""Thyroid gland"",""Others"",""gkmSVM""}]",null,null,-0.368421,"""ELA""","""255,202,74""","""yellow"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 3. Selection of transcript (DOWN-DOWN)

- 上記のセルで読み込んだ enhancer 側の transcript と、target promoter 側の transcript をそれぞれ candidate として選択する
- enhancer 側の transcript と、target promoter 側の transcript はこの時点では同じ数になり、join することによってフィルタリングしていく

In [5]:
# enhancer side transcript entry (enhancer potential transcript - CRE pair)
fantabio_HN2_down5p_enhancer_transcript_candidate = down5p_fantabio.select(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "transcript_id",
    "HN2",
    "transcript_type",
    "gene_id",
    "gene_symbol",
    "strand",
    "RNACentral_ID",
    "annotation_remarks",
    "tags"
).unique().rename(
    # assign frag name "e" to enhancer side
    {
        "transcript_id": "e_transcript_id",
        "HN2": "e_HN2",
        "transcript_type": "e_transcript_type",
        "gene_id": "e_gene_id",
        "gene_symbol": "e_gene_symbol",
        "strand": "e_strand",
        "RNACentral_ID": "e_RNACentral_ID",
        "annotation_remarks": "e_annotation_remarks",
        "tags": "e_tags"
    }
)

display(fantabio_HN2_down5p_enhancer_transcript_candidate)

cre_id,cre_name,fantabio_cre_class,e_transcript_id,e_HN2,e_transcript_type,e_gene_id,e_gene_symbol,e_strand,e_RNACentral_ID,e_annotation_remarks,e_tags
str,str,str,str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_168202""","""cp6@SNHG29""","""ELA""","""ENST00000480811.6""",-105,"""lncRNA""","""ENSG00000175061.21""","""SNHG29""","""+""","""URS0000EF50B5""","[""Assembled from PacBio CLS reads - CAGE_PolyA_HiSeq (GSE93848)""]","[""TAGENE""]"
…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_23877""","""cp4@LINC01138""","""ELA""","""ENST00000614292.2""",-125,"""lncRNA""","""ENSG00000274020.6""","""LINC01138""","""-""","""URS0002A59B74""","[""CLS3 project""]","[""TAGENE""]"


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## JOIN (1)

- 標的となる promoter の近傍にある transcript は一旦統合せず、enhancer RNA の potential がある transcript だけを一旦出力する

In [7]:
e2g_join_filter_unique_count_join = e2g_join_filter_unique_count.join(
    fantabio_HN2_down5p_enhancer_transcript_candidate,
    on=["cre_id", "cre_name", "fantabio_cre_class"],
    how="inner"
).filter(
    pl.col("e_transcript_type") == "lncRNA" # lncRNA のみを対象とする
)


display(e2g_join_filter_unique_count_join)

cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean,e_transcript_id,e_HN2,e_transcript_type,e_gene_id,e_gene_symbol,e_strand,e_RNACentral_ID,e_annotation_remarks,e_tags
str,str,str,str,str,str,list[cat],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64,str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_348363""","""FCHS_348363""","""PLA""","""FCHS_348367""","""cp5@PRP4K""","""PLA""","[""ENCSR000MZA_ENCFF654AMU"", ""ENCSR002YWN_ENCFF435DPV"", … ""ENCSR999YQC_ENCFF625FFH""]",1459,"[0.860104, 0.863558, … 0.849741]",0.702936,1.0,0.845307,"[0.994501, 0.981009, … 0.95402]",0.363008,0.996145,0.981076,"""ENST00000606564.1""",-52,"""lncRNA""","""ENSG00000272248.2""",null,"""+""","""URS00004E6312""",null,"[""basic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_183794""","""cp2@LOC105371912""","""PLA""","""FCHS_186267""","""cp14@METRNL""","""PLA""","[""ENCSR621BIJ_ENCFF647VTW""]",1,[1.0],1.0,1.0,1.0,[0.612981],0.612981,0.612981,0.612981,"""ENST00000586321.1""",-68,"""lncRNA""","""ENSG00000267737.2""",null,"""+""","""URS00004288D2""",null,"[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]"


In [8]:
e2g_join_filter_unique_count_join.write_parquet(
    "../../out/transcript_classification/enhancer_derived_lncRNA/down5p_enhancer_derived_lncRNA_v261010.parquet"
)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Join (2) (DOWN (enhancer proximal lncRNA) - DOWN (target promoter proximal transcript))

- このプロセスで `join`することで得られるのは、enhancer - promoter 両方の近傍にある transcript が発現低下したとみなしている transcript
- 予測された enhancer 側で、`PLA` or `ELA` でもフィルタリングしないことに決定


In [9]:
# target promoter side transcript candidate
fantabio_HN2_down5p_target_transcript_candidate = down5p_fantabio.select(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "transcript_id",
    "HN2",
    "transcript_type",
    "gene_id",
    "gene_symbol",
    "strand",
    "RNACentral_ID",
    "annotation_remarks",
    "tags"
).unique().rename(
    # assign frag name "t" to target promoter side
    {
        "cre_id": "target_cre_id",
        "cre_name": "target_cre_name",
        "fantabio_cre_class": "target_fantabio_cre_class",
        "transcript_id": "t_transcript_id",
        "HN2": "t_HN2",
        "transcript_type": "t_transcript_type",
        "gene_id": "t_gene_id",
        "gene_symbol": "t_gene_symbol",
        "strand": "t_strand",
        "RNACentral_ID": "t_RNACentral_ID",
        "annotation_remarks": "t_annotation_remarks",
        "tags": "t_tags"
    }
)

display(fantabio_HN2_down5p_target_transcript_candidate)

target_cre_id,target_cre_name,target_fantabio_cre_class,t_transcript_id,t_HN2,t_transcript_type,t_gene_id,t_gene_symbol,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags
str,str,str,str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_166929""","""cp8@PFAS""","""PLA""","""ENST00000868169.1""",-48,"""protein_coding""","""ENSG00000178971.18""","""CTC1""","""-""",null,"[""DanaFarber""]","[""basic"", ""GENCODE_Primary"", ""TAGENE""]"
…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_348368""","""cp6@PRP4K""","""ELA""","""ENST00000935173.1""",-177,"""protein_coding""","""ENSG00000112739.18""","""PRP4K""","""+""",null,"[""LRGASP""]","[""basic"", ""GENCODE_Primary"", … ""appris_alternative_2""]"


In [10]:
e2g_join_filter_unique_count_join_down_down = e2g_join_filter_unique_count.join(
    fantabio_HN2_down5p_enhancer_transcript_candidate,
    on=["cre_id", "cre_name", "fantabio_cre_class"],
    how="inner"
).join(
    fantabio_HN2_down5p_target_transcript_candidate,
    on=["target_cre_id", "target_cre_name", "target_fantabio_cre_class"],
    how="inner"
).sort(
    by=["e_HN2", "t_HN2", "re2g_dataset_count"],
    descending=[True, True, True]
).filter(
    pl.col("e_transcript_type") == "lncRNA" # lncRNA のみを対象とする
)

e2g_join_filter_unique_count_join_down_down.write_parquet(
    "../../out/transcript_classification/enhancer_derived_lncRNA/down5p_enhancer_derived_lncRNA_v261010_DOWN_DOWN.parquet"
)

display(e2g_join_filter_unique_count_join_down_down)


cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean,e_transcript_id,e_HN2,e_transcript_type,e_gene_id,e_gene_symbol,e_strand,e_RNACentral_ID,e_annotation_remarks,e_tags,t_transcript_id,t_HN2,t_transcript_type,t_gene_id,t_gene_symbol,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags
str,str,str,str,str,str,list[cat],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64,str,i64,str,str,str,str,str,list[str],list[str],str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_283074""","""cp7@SLC25A38""","""PLA""","""FCHS_283071""","""cp4@SLC25A38""","""PLA""","[""ENCSR002YWN_ENCFF435DPV"", ""ENCSR004ZDG_ENCFF053LFW"", … ""ENCSR998QCO_ENCFF829OYF""]",895,"[0.953935, 0.942418, … 0.952015]",0.819578,1.0,0.952341,"[0.993512, 0.991018, … 0.824871]",0.312936,0.998512,0.971214,"""ENST00000836881.1""",-48,"""lncRNA""","""ENSG00000287780.2""",null,"""-""","""URS0002A51F68""","[""CLS3 project""]","[""TAGENE""]","""ENST00000836881.1""",-48,"""lncRNA""","""ENSG00000287780.2""",null,"""-""","""URS0002A51F68""","[""CLS3 project""]","[""TAGENE""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_173068""","""cp2@SMARCE1""","""ELA""","""FCHS_173069""","""cp6@SMARCE1""","""PLA""","[""ENCSR226FNO_ENCFF126EEY""]",1,[0.508507],0.508507,0.508507,0.508507,[0.579657],0.579657,0.579657,0.579657,"""ENST00000619697.3""",-198,"""lncRNA""","""ENSG00000278834.3""",null,"""+""","""URS0002A54E6E""","[""CLS3 project"", ""Assembled from PacBio CLS3 reads""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""ENST00000619697.3""",-198,"""lncRNA""","""ENSG00000278834.3""",null,"""+""","""URS0002A54E6E""","[""CLS3 project"", ""Assembled from PacBio CLS3 reads""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]"
